In [ ]:
import pandas as pd

df_deter = pd.read_csv("deter_limpio_merge.csv")

centroides = (
    df_deter
    .groupby(["municipio_norm", "uf"])
    .agg(lat=("lat", "mean"), lon=("lon", "mean"))
    .reset_index()
)

print("Municipios únicos:", centroides.shape[0])
print(centroides.head(10))

Municipios únicos: 425
       municipio_norm  uf       lat        lon
0          ABAETETUBA  PA -1.865378 -48.772760
1     ABEL FIGUEIREDO  PA -4.865440 -48.488341
2          ACAILANDIA  MA -4.735591 -47.663456
3               ACARA  PA -2.280823 -48.489029
4          ACRELANDIA  AC -9.920059 -66.905459
5                AFUA  PA -0.198676 -51.161344
6  AGUA AZUL DO NORTE  PA -6.647725 -50.612804
7            ALENQUER  PA -1.531425 -54.982492
8            ALMEIRIM  PA -1.224039 -53.162442
9       ALTA FLORESTA  MT -9.918474 -56.330947


In [ ]:
import requests

fila_prueba = centroides.iloc[0]
lat, lon = fila_prueba["lat"], fila_prueba["lon"]

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,RH2M,WS2M",   # Temperatura 2m, Humedad relativa 2m, Velocidad de viento 2m
    "community": "AG",
    "longitude": lon,
    "latitude": lat,
    "start": "20220101",
    "end": "20241231",
    "format": "JSON"
}

response = requests.get(url, params=params)
print("Status code:", response.status_code)

data = response.json()
print(data.keys())

Status code: 200
dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])


In [ ]:
print(data["properties"].keys())

parametros = data["properties"]["parameter"]
print(parametros.keys())

dict_keys(['parameter'])
dict_keys(['T2M', 'RH2M', 'WS2M'])


In [ ]:
t2m = parametros["T2M"]
primeros_5 = dict(list(t2m.items())[:5])
print(primeros_5)

{'20220101': 25.62, '20220102': 25.8, '20220103': 25.34, '20220104': 25.45, '20220105': 25.75}


In [ ]:
import numpy as np

def json_a_dataframe(data, municipio_norm, uf):
    """Convierte la respuesta de NASA POWER en un DataFrame con una fila por día."""
    parametros = data["properties"]["parameter"]

    df_temp = pd.DataFrame(parametros)  # cada columna es T2M, RH2M, WS2M; el índice son las fechas (texto)
    df_temp = df_temp.reset_index().rename(columns={"index": "fecha_str"})

    df_temp["fecha"] = pd.to_datetime(df_temp["fecha_str"], format="%Y%m%d")
    df_temp["municipio_norm"] = municipio_norm
    df_temp["uf"] = uf

    df_temp = df_temp.drop(columns=["fecha_str"])

    # reemplazar el código de relleno -999 por NaN, igual que hicimos con BDQueimadas
    for col in ["T2M", "RH2M", "WS2M"]:
        df_temp[col] = df_temp[col].replace(-999, np.nan)

    return df_temp

# probamos con el municipio de prueba (ABAETETUBA)
df_prueba = json_a_dataframe(data, fila_prueba["municipio_norm"], fila_prueba["uf"])

print(df_prueba.shape)
print(df_prueba.dtypes)
print(df_prueba.head())

(1096, 6)
T2M                      float64
RH2M                     float64
WS2M                     float64
fecha             datetime64[ns]
municipio_norm            object
uf                        object
dtype: object
     T2M   RH2M  WS2M      fecha municipio_norm  uf
0  25.62  92.18  0.14 2022-01-01     ABAETETUBA  PA
1  25.80  90.77  0.11 2022-01-02     ABAETETUBA  PA
2  25.34  93.46  0.08 2022-01-03     ABAETETUBA  PA
3  25.45  88.97  0.17 2022-01-04     ABAETETUBA  PA
4  25.75  90.75  0.06 2022-01-05     ABAETETUBA  PA


In [ ]:
import time

resultados = []
errores = []

for i, row in centroides.iterrows():
    try:
        params = {
            "parameters": "T2M,RH2M,WS2M",
            "community": "AG",
            "longitude": row["lon"],
            "latitude": row["lat"],
            "start": "20220101",
            "end": "20241231",
            "format": "JSON"
        }
        response = requests.get(url, params=params, timeout=30)

        if response.status_code == 200:
            data_municipio = response.json()
            df_municipio = json_a_dataframe(data_municipio, row["municipio_norm"], row["uf"])
            resultados.append(df_municipio)
        else:
            errores.append((row["municipio_norm"], row["uf"], response.status_code))

    except Exception as e:
        errores.append((row["municipio_norm"], row["uf"], str(e)))

    if (i + 1) % 50 == 0:
        print(f"Progreso: {i + 1}/{len(centroides)} municipios procesados")

    time.sleep(0.3)  # pausa breve para no saturar la API

print(f"\nCompletado: {len(resultados)} municipios exitosos, {len(errores)} con error")

Progreso: 50/425 municipios procesados
Progreso: 100/425 municipios procesados
Progreso: 150/425 municipios procesados
Progreso: 200/425 municipios procesados
Progreso: 250/425 municipios procesados
Progreso: 300/425 municipios procesados
Progreso: 350/425 municipios procesados
Progreso: 400/425 municipios procesados

Completado: 425 municipios exitosos, 0 con error


In [ ]:
df_nasa = pd.concat(resultados, ignore_index=True)

print("Shape combinado:", df_nasa.shape)
print(df_nasa.dtypes)
df_nasa.head()

Shape combinado: (465800, 6)
T2M                      float64
RH2M                     float64
WS2M                     float64
fecha             datetime64[ns]
municipio_norm            object
uf                        object
dtype: object


,T2M,RH2M,WS2M,fecha,municipio_norm,uf
0,25.62,92.18,0.14,2022-01-01,ABAETETUBA,PA
1,25.80,90.77,0.11,2022-01-02,ABAETETUBA,PA
2,25.34,93.46,0.08,2022-01-03,ABAETETUBA,PA
3,25.45,88.97,0.17,2022-01-04,ABAETETUBA,PA
4,25.75,90.75,0.06,2022-01-05,ABAETETUBA,PA


In [ ]:
print("Valores nulos por columna (donde había -999):")
print(df_nasa.isnull().sum())

print("\nEstadísticas de las 3 variables:")
print(df_nasa[["T2M", "RH2M", "WS2M"]].describe())

print("\nRango de fechas:", df_nasa["fecha"].min(), "→", df_nasa["fecha"].max())
print("Municipios únicos:", df_nasa["municipio_norm"].nunique())

Valores nulos por columna (donde había -999):
T2M               0
RH2M              0
WS2M              0
fecha             0
municipio_norm    0
uf                0
dtype: int64

Estadísticas de las 3 variables:
                T2M           RH2M           WS2M
count  465800.00000  465800.000000  465800.000000
mean       27.28461      75.514555       0.481639
std         2.54798      17.658214       0.676754
min        10.52000      13.540000       0.000000
25%        25.54000      63.450000       0.060000
50%        26.73000      82.510000       0.170000
75%        28.87000      89.700000       0.650000
max        38.23000      99.180000       8.320000

Rango de fechas: 2022-01-01 00:00:00 → 2024-12-31 00:00:00
Municipios únicos: 424


In [ ]:
print("Combinaciones únicas (municipio_norm, uf):", df_nasa[["municipio_norm", "uf"]].drop_duplicates().shape[0])

Combinaciones únicas (municipio_norm, uf): 425


In [ ]:
df_nasa.to_csv("nasa_power_limpio.csv", index=False, encoding="utf-8")
print("nasa_power_limpio.csv guardado:", df_nasa.shape)

nasa_power_limpio.csv guardado: (465800, 6)
